# 01 · `landing-zone` (CSV) → `bronze` (Delta Lake)

Lê os CSVs do bucket `landing-zone` e grava cada tabela em formato **Delta Lake** no bucket `bronze`.

```mermaid
flowchart LR
    A[(MinIO<br/>landing-zone<br/>CSV)] --> B[Spark]
    B -->|Delta| C[(MinIO<br/>bronze<br/>Delta Lake)]
```

In [1]:
from spark_delta_minio_sqlserver import pipeline
from spark_delta_minio_sqlserver.spark_session import create_session

spark = create_session("01-landing-para-bronze")
spark.sparkContext.setLogLevel("ERROR")

26/10/11 12:44:24 WARN Utils: Your hostname, NF-ENG-ISABELLEFELTRIN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/11 12:44:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/isafel/.ivy2/cache
The jars for the packages stored in: /home/isafel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6db0fd89-4423-4a25-8629-dcb5e023224b;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/isafel/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 in central


:: resolution report :: resolve 269ms :: artifacts dl 25ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 from central in [default]
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   7   |   0   |   0   |   0   ||   7   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
TABELAS = pipeline.listar_landing()
pipeline.landing_para_bronze(spark, TABELAS)

{'marca': 6, 'cliente': 4, 'carro': 8, 'apolice': 8}

## Conferência: tabelas Delta no bucket `bronze`

In [3]:
for tabela in TABELAS:
    print(tabela)
    spark.read.format("delta").load(f"s3a://bronze/{tabela}").show(3)

marca


+---+-------+
| id|   nome|
+---+-------+
|  1|RENAULT|
|  2|HYUNDAI|
|  3|   FIAT|
+---+-------+
only showing top 3 rows

cliente


+---+-------------+-----------+-------------+
| id|         nome|        cpf|       cidade|
+---+-------------+-----------+-------------+
|  1|    Ana Souza|11111111111|     Criciúma|
|  2|   Bruno Lima|22222222222|Florianópolis|
|  3|Carla Pereira|33333333333|      Tubarão|
+---+-------------+-----------+-------------+
only showing top 3 rows

carro


+-------+------+-----------+--------+----+------+----------+
|  placa|modelo|     chassi|marca_id| ano|   cor|cliente_id|
+-------+------+-----------+--------+----+------+----------+
|ALD3834|  CLIO|34574215969|       1|2011|BRANCO|         1|
|CCR8096| CRETA|88547875547|       2|2020|BRANCO|         1|
|DLA3438| PUNTO|98823483434|       3|2013| PRETO|         2|
+-------+------+-----------+--------+----+------+----------+
only showing top 3 rows

apolice


+---+-------+------+-----------+
| id|  placa| valor|data_inicio|
+---+-------+------+-----------+
|  1|ALD3834|1850.0| 2024-01-10|
|  2|CCR8096|2790.5| 2024-02-15|
|  3|DLA3438|1620.0| 2024-03-01|
+---+-------+------+-----------+
only showing top 3 rows



In [4]:
spark.read.format("delta").load("s3a://bronze/apolice").printSchema()

root
 |-- id: integer (nullable = true)
 |-- placa: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- data_inicio: date (nullable = true)



Cada tabela tem agora o diretório `_delta_log/` junto aos arquivos Parquet. Veja em http://127.0.0.1:9001.